# VietMedBridge — 04: BGE + BM25 trên corpus lớn / full inference cho pilot

Plan chính: `R2AI_STAGE3_FULL_COMPETITION_AND_BEST_OF_STAGE1_STAGE2.md`.
Chọn **runtime GPU mới, T4 trở lên**, rồi Run all. Giữ DATA_ROOT cũ;
không chạy lại 00–03 và không chọn ACTION. Bootstrap tự clone/cài package.

Nếu 03 đã publish active_data_candidate.json, tự nhận candidate đó.
Corpus vượt giới hạn pilot (2.000 documents hoặc 50.000 children) chạy
**baseline lấy điểm: BGE-M3 toàn bộ child inputs + BM25 VI/EN/ZH trên disk
→ weighted RRF → frozen source parents + LCS dedup → ZIP đủ 1.200 query**.
Tự nhận team-100k-data-v1, không quay về corpus 864 documents cũ.
BGE lưu vector theo 141 phần input của 03, sort theo độ dài khi encode rồi
khôi phục đúng thứ tự. Batch khởi đầu 32, tự giảm khi OOM. Tìm dense quét
toàn bộ vector parts theo blocks trên GPU; source/BM25 ở SQLite local,
checkpoint đầy đủ và ZIP nằm trên Drive. Không nạp toàn corpus vào RAM.

Benchmark T4 của Sếp: BGE batch 4 ~46 texts/s (child projection ~3,45h),
Qwen embedding batch 4 ~6,65 texts/s (~24h). Đây chỉ là mẫu 64 texts;
batch 32 chưa có phép đo thực tế, không bảo đảm thời gian hoặc điểm.
Corpus lớn dùng BGE + BM25 trước; **chưa phải full master plan**:
Qwen embedding, query LLM, neural reranker, document-dense và adaptive
parents được hoãn. Dùng frozen parents đã kiểm source từ 03.
Không cần chạy lại 02–03 hoặc chạy 05 cho lượt baseline này.

Kiến trúc full cho pilot nhỏ: **BGE-M3 child/document dense + Qwen3-Embedding-8B + BM25 VI/EN/ZH → weighted RRF
→ Qwen3-Reranker-8B document → local child retrieval/MaxP rerank
→ exact-source parent 512/640 → LCS dedup → ZIP đủ 1.200 query**.
Qwen3-4B dịch query; PICO-lite/subqueries/HyDE chỉ additive cho query phức tạp.
Các số/Latin entities/constraint cues được kiểm; original query luôn giữ.
Sparse dùng PyVI, Jieba, CJK n-grams và title/heading/body/alias fields.

Sếp đã xác nhận với BTC: **≤15B cho từng model**, không tính tổng model.
BGE-M3 0,568B; Qwen embedding 7,567B; query LLM 4,022B; reranker 8,189B.
Gate tính adapter vào base model tương ứng, trước quantization.
Hai model 8B dùng **NF4 4-bit**.
Models nạp lần lượt. Đây là lựa chọn tài nguyên
cho T4, chưa benchmark tương đương fp16. Weights/cache nằm ổ local Colab;
vector parts, translation/expansion và scored stages checkpoint trên Drive.
Khi Colab ngắt, Run all lại cùng cấu hình; không chạy hai runtime cùng run.

Với corpus nhỏ: chạy full inference và xuất submission như trước.
Có code train/dev mining, QLoRA, checkpoint selection, dev F2 cutoff và
ablation trong module/05. **Hiện chưa có train/dev labels** nên 04 dùng
pretrained models và cutoff chưa tune; không giả lập fine-tune hoặc điểm.
Nếu đã chạy 05 với nhãn độc lập hợp lệ, 04 tự đọc adapter/dev policy đã chọn,
tạo namespace -ft- mới. Nguồn/candidate cũ và các kết quả baseline được giữ.

Với pilot: reuse verified corpus/document vectors từ cả 04 và 05. Chỉ encode
inputs còn thiếu. Batch Qwen tự chọn theo VRAM và giảm nếu OOM.
Theo dõi runtime_profile.json/execution_policy.json trong thư mục run;
mỗi embedding pass cũng ghi thời gian encode, số parts mới/đã cache.
Model revisions của full pilot giữ nguyên.


## 1. Bootstrap, mount Drive, clone và cài dependencies — CPU


In [ ]:
import importlib.util
import json
import os
import subprocess
import sys
from pathlib import Path

if not (3, 11) <= sys.version_info[:2] < (3, 14):
    raise RuntimeError(
        "Dùng Python 3.11–3.13. Trong Runtime > Change runtime type, "
        "chọn phiên bản runtime có Python trong khoảng này."
    )
if importlib.util.find_spec("google.colab") is None:
    raise RuntimeError("Notebook này được thiết kế cho Google Colab.")

from google.colab import drive
# Chạy lại cell này sau MỖI runtime mới/restart: package chỉ được cài trong phiên Colab.
drive.mount("/content/drive")

# Giữ DATA_ROOT đã dùng ở notebook 00–03.
DATA_ROOT = Path("/content/drive/MyDrive/VietMedBridge/data")
WORK_DIR = Path("/content/vmb_work")  # temp files/spill ở ổ local của Colab
DATA_ROOT.mkdir(parents=True, exist_ok=True)
WORK_DIR.mkdir(parents=True, exist_ok=True)
os.environ["HF_HOME"] = "/content/hf_cache"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

REPO_URL = "https://github.com/Platypus27-coder/VietMedBridge.git"
CODE_REVISION = None  # Có thể đặt full commit SHA; mặc định dùng code_lock đã lưu.
EXPECTED_PIPELINE_API = 2
CHECKOUT = Path("/content/VietMedBridge")
lock_path = DATA_ROOT / "retrieval_code_lock.json"
lock = json.loads(lock_path.read_text()) if lock_path.exists() else {}
if lock and lock.get("repo_url") != REPO_URL:
    raise ValueError("DATA_ROOT đang khóa vào một repo khác.")
if lock and lock.get("pipeline_api") != EXPECTED_PIPELINE_API and not CODE_REVISION:
    raise RuntimeError(
        "DATA_ROOT đang dùng code cũ. Để nâng lên data-v2, đặt CODE_REVISION='main' "
        "ở cell này rồi chạy lại; dùng tên crawl/build run mới. Raw cũ vẫn được giữ."
    )
RETRIEVAL_WORKFLOW_API = "full-master-plan-strong-v6-large-baseline"
upgrade = lock.get("workflow_api") != RETRIEVAL_WORKFLOW_API
reference = CODE_REVISION or ("main" if upgrade else lock.get("git_commit")) or "main"
if not (CHECKOUT / ".git").exists():
    subprocess.run(["git", "clone", "--no-checkout", REPO_URL, str(CHECKOUT)], check=True)
remote = subprocess.check_output(
    ["git", "-C", str(CHECKOUT), "remote", "get-url", "origin"], text=True
).strip()
if remote != REPO_URL:
    raise ValueError("Checkout hiện tại không thuộc repo VietMedBridge.")
subprocess.run(["git", "-C", str(CHECKOUT), "fetch", "--depth", "1", "origin", reference], check=True)
CODE_COMMIT = subprocess.check_output(
    ["git", "-C", str(CHECKOUT), "rev-parse", "FETCH_HEAD"], text=True
).strip()
if "vietmedbridge" in sys.modules and globals().get("_VMB_IMPORTED_COMMIT") != CODE_COMMIT:
    raise RuntimeError("Mã nguồn đổi trong runtime đã import package. Restart session rồi chạy lại notebook.")
subprocess.run(["git", "-C", str(CHECKOUT), "checkout", "--detach", "FETCH_HEAD"], check=True)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--disable-pip-version-check", "-e", ".[notebook,retrieval,strong]"],
    cwd=CHECKOUT,
    check=True,
)
PACKAGE_SOURCE = CHECKOUT / "src"
if not (PACKAGE_SOURCE / "vietmedbridge" / "__init__.py").is_file():
    raise RuntimeError(
        f"Repo checkout thiếu package source: {PACKAGE_SOURCE}; commit={CODE_COMMIT}. "
        "Kiểm tra lại git fetch/checkout ở output của cell Bootstrap."
    )
sys.path.insert(0, str(PACKAGE_SOURCE))
importlib.invalidate_caches()
from vietmedbridge.artifacts import atomic_json, runtime_versions
from vietmedbridge import PIPELINE_API_VERSION
if PIPELINE_API_VERSION != EXPECTED_PIPELINE_API:
    raise RuntimeError("Notebook và package API không khớp; chọn commit data-v2 và restart session.")
_VMB_IMPORTED_COMMIT = CODE_COMMIT
if not lock or CODE_REVISION or upgrade:
    atomic_json(lock_path, {"repo_url": REPO_URL, "git_commit": CODE_COMMIT, "pipeline_api": PIPELINE_API_VERSION, "workflow_api": RETRIEVAL_WORKFLOW_API})
PIPELINE_CONFIG = json.loads((CHECKOUT / "configs/data_pipeline.json").read_text())
atomic_json(DATA_ROOT / "retrieval_runtime.json", {
    "git_commit": CODE_COMMIT, "python": sys.version, "packages": runtime_versions()
})
print("Code commit:", CODE_COMMIT)
import vietmedbridge
print("Package source:", Path(vietmedbridge.__file__).resolve())
print("Dữ liệu/checkpoint:", DATA_ROOT)


## 2. Chạy baseline corpus lớn hoặc full pilot — CPU/GPU lần lượt

Giữ MAX_NEW_*=None để chạy hết. Đổi model/data/policy cần RUN_NAME mới;
các run cũ không bị ghi đè. Vector BGE của stage-a-retrieval-v1 được kiểm
hash/order/model trước reuse. Qwen vectors có corpus/query identities riêng.

Sau checks nguồn/snapshot/query, GPU chạy phần thiếu:
BGE child/document → Qwen4B query understanding → Qwen8B embedding → Qwen8B rerank.
Run v3-per-model-15b tách khỏi v1/v2; không trộn Qwen vectors 0.6B và 8B.
Giữ DATA_ROOT; corpus và BGE vectors hợp lệ vẫn được tận dụng.
Runtime lưu từng phần; lỗi mạng/OOM giữ checkpoints đã hoàn tất.
Với candidate lớn: chỉ BGE + BM25; MAX_NEW_EMBEDDING_PARTS và
MAX_NEW_QUERIES có thể giới hạn mỗi phiên, mặc định None chạy hết.
Ngắt runtime thì mở cùng notebook và Run all để resume vector/query.
Catalog CPU xuất bản atomic; nếu ngắt trước hoàn tất catalog thì dựng
lại riêng bước CPU đó. Không chạy hai runtime ghi cùng run.
Glossary tùy chọn data/labels/medical_aliases.json cần reviewed=true,
entries=[{aliases:[...], source:"..."}]; không có thì alias field rỗng,
biomedical Latin/acronym tokens vẫn được giữ.


In [ ]:
import json
from vietmedbridge.full_plan_runtime import run_full_pipeline
from vietmedbridge.scale_benchmark import load_handoff
from vietmedbridge.scale_baseline import run_large_baseline
from vietmedbridge.artifacts import read_json

BUILD_RUN = "stage-a-data-v3-laodong"
CANDIDATE_NAME = "candidate-1cd220a4be956d5a.json"
RUN_NAME = "stage-a-full-plan-v3-per-model-15b"
EMBEDDING_CACHE_RUN = "stage-a-retrieval-v1"
MAX_NEW_EMBEDDING_PARTS = None
MAX_NEW_TRANSLATIONS = None
MAX_NEW_QUERIES = None
BGE_BATCH_SIZE = 32  # execution only; tự giảm nếu OOM, không đổi source/model

LARGE_CANDIDATE = False
if (DATA_ROOT / "active_data_candidate.json").is_file():
    active = read_json(DATA_ROOT / "active_data_candidate.json")
    _, candidate, _ = load_handoff(DATA_ROOT)
    BUILD_RUN, CANDIDATE_NAME = active["build_run"], active["candidate_name"]
    limits = read_json(CHECKOUT / "configs/retrieval_full.json")["pilot_limits"]
    LARGE_CANDIDATE = (candidate["counts"]["documents"] > limits["max_documents"]
        or candidate["counts"]["children"] > limits["max_children"])
    RUN_NAME = BUILD_RUN + "-full-plan-v1"
if LARGE_CANDIDATE:
    RESULT = run_large_baseline(DATA_ROOT, CHECKOUT, code_commit=CODE_COMMIT,
        work_dir=WORK_DIR, batch_size=BGE_BATCH_SIZE,
        max_new_embedding_parts=MAX_NEW_EMBEDDING_PARTS, max_new_queries=MAX_NEW_QUERIES)
    READY = RESULT["ready"]
else:
    RESULT = run_full_pipeline(DATA_ROOT, CHECKOUT, code_commit=CODE_COMMIT,
        work_dir=WORK_DIR, build_run=BUILD_RUN, candidate_name=CANDIDATE_NAME,
        run_name=RUN_NAME, embedding_cache_run=EMBEDDING_CACHE_RUN,
        max_new_embedding_parts=MAX_NEW_EMBEDDING_PARTS,
        max_new_translations=MAX_NEW_TRANSLATIONS, max_new_queries=MAX_NEW_QUERIES)
    READY = RESULT["ready"]
print(json.dumps(RESULT["status"], ensure_ascii=False, indent=2))
print(json.dumps(READY, ensure_ascii=False, indent=2))


## 3. Xem source samples và báo cáo — CPU

Chunks bên dưới là source slices. Với corpus lớn, baseline_status.json
ghi đúng BGE + BM25, số documents/children và các phần full plan đang hoãn;
runtime_profile.json ghi thời gian từng stage. Với full pilot, diagnostics.json
có language coverage, branch contributions, output counts và logit distributions. Không có nhãn
thì không tính F2 giả. full_plan_status.json phân biệt code đã thực thi với
supervised/scale gates còn cần dữ liệu và đo đạc. Corpus nhỏ và chất lượng
nguồn vẫn giới hạn recall.


In [ ]:
for sample in RESULT["samples"]:
    print("\nQuery", sample["query"]["id"], sample["query"]["query"])
    prediction = sample["prediction"]
    print("Documents:", prediction["relevant_docs"])
    print("First chunk:", prediction["relevant_chunks"][0]["chunk_text"][:800]
          if prediction["relevant_chunks"] else "EMPTY")
print("Diagnostics:", RESULT["diagnostics_path"])


## 4. Tải ZIP và submit thử để lấy điểm thật

ZIP chỉ chứa results.json ở root, đủ 1.200 official IDs; kiểm source/schema
và checksum trước download. Manifest/model/runtime evidence nằm ngoài ZIP.
Upload ZIP lên Dashboard BTC; score_feedback.json giữ submission ID và
điểm thật, không bị reset khi resume. Chưa có official score trước upload.


In [ ]:
from google.colab import files
from vietmedbridge.artifacts import verify_file
if READY["state"] != "READY_FOR_MANUAL_UPLOAD" or READY["query_count"] != 1200:
    print("Đã lưu checkpoint:", READY)
    print("Run all lại cùng cấu hình để hoàn tất trước download.")
else:
    verify_file(READY["zip_path"], READY["zip_sha256"])
    files.download(READY["zip_path"])
